In [1]:
import os

print("CPU count:", os.cpu_count())

CPU count: 11


In [2]:
from src.dataset.tiny_imagenet_dataset_2 import TinyImageNetLoader
SEED = 42
train_dataset, val_dataset, _, _ = TinyImageNetLoader(seed=SEED).load()

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Training images: 100000
Validation images: 10000


In [5]:
import time
from torch.utils.data import DataLoader


def benchmark_workers(dataset, batch_size=64, worker_values=(0, 2, 4), num_batches=200, repeats=3):
    for workers in worker_values:
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=workers,
            persistent_workers=workers > 0
        )

        times = []

        for repeat in range(repeats):
            start = time.time()

            for batch_idx, (images, labels) in enumerate(loader):
                if batch_idx + 1 >= num_batches:
                    break

            elapsed = time.time() - start
            times.append(elapsed)

        avg_time = sum(times) / len(times)
        batches_per_sec = num_batches / avg_time

        print(
            f"Workers: {workers:<2} | "
            f"Times: {[round(t, 2) for t in times]} | "
            f"Average: {avg_time:.2f}s | "
            f"{batches_per_sec:.2f} batches/sec"
        )

In [7]:
benchmark_workers(
    train_dataset,
    batch_size=64,
    worker_values=(0, 2, 4, 6, 8),
    num_batches=200,
    repeats=3
)

Workers: 0  | Times: [5.8, 5.79, 5.67] | Average: 5.75s | 34.77 batches/sec
Workers: 2  | Times: [5.53, 2.8, 2.77] | Average: 3.70s | 54.04 batches/sec
Workers: 4  | Times: [5.82, 1.78, 1.52] | Average: 3.04s | 65.79 batches/sec
Workers: 6  | Times: [7.82, 1.28, 1.25] | Average: 3.45s | 57.99 batches/sec
Workers: 8  | Times: [9.85, 1.22, 1.19] | Average: 4.09s | 48.95 batches/sec
